# Notebook 69 — Minimal Basis Refinement

Notebook 68 showed the six-term field is strong and near-minimal, but exposed one wrinkle:

> The constant term helps pointwise field RMSE but can slightly hurt trajectory RMSE by introducing integration drift.

Notebook 69 answers:

> Should the final paper use the raw six-term field, a centered-constant variant, a reduced pairwise basis, or an orthogonalized validation basis?

Base raw field:

\[
g_0(r,c;\beta)
=
\beta_0+\beta_c c+\beta_r r+\beta_{c^3}c^3+\beta_{r^2}r^2+\beta_{rc^2}rc^2
\]

Tests:
1. Constant variants: raw, no constant, centered constant, centered all terms.
2. Pairwise ablation: weak/refinement pairs.
3. Orthogonalized basis: QR validation for basis correlation.
4. Final verdict table and paper-ready paragraph.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, glob, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LassoCV, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, LeaveOneOut

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda x: x

np.random.seed(42)
OUTPUT_DIR = "paper69_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True, "grid.alpha": 0.25})

In [ ]:
# Data loading and synthetic fallback

DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = ["*residual*flow*.parquet", "*residual*flow*.csv", "*governing*flow*.parquet", "*governing*flow*.csv", "*.parquet", "*.csv", "*.pkl", "*.pickle"]
    for pat in patterns:
        candidates += glob.glob(pat)
        candidates += glob.glob(os.path.join("data", pat))
        candidates += glob.glob(os.path.join("outputs", pat))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        c_grid = np.linspace(-1.25, 1.05, 42)
                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.05, 5: 0.02, 7: 0.06}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(14):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                dc = c_grid[min(window_id + 1, len(c_grid)-1)] - c if window_id < len(c_grid)-1 else c - c_grid[max(window_id-1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * dc
                                rows.append({"system": system, "task": task, "forcing_mode": forcing_mode, "k": k, "flow_mode": flow_mode, "condition_coord": c, "residual": r, "predicted_flow": g + noise, "next_residual": next_r, "delta_condition": dc, "sample_id": sample_id, "path_id": path_id, "window_id": window_id})
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

def align_schema(df):
    alias_groups = {
        "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
        "residual": ["residual", "r", "resid"],
        "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
        "next_residual": ["next_residual", "r_next", "next_r"],
        "delta_condition": ["delta_condition", "dc", "delta_c"],
        "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
    }
    rename = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in df.columns and a != canonical:
                rename[a] = canonical
                break
    df = df.rename(columns=rename)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]
    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        df["delta_condition"] = np.gradient(df["condition_coord"].to_numpy())

    defaults = {"system": "unknown_system", "task": "unknown_task", "forcing_mode": "unknown_forcing", "k": 5, "flow_mode": "unknown_flow", "sample_id": np.arange(len(df)), "path_id": 0, "window_id": np.arange(len(df))}
    for key, val in defaults.items():
        if key not in df.columns:
            df[key] = val

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Synthetic fallback:", USE_SYNTHETIC_FALLBACK)
print("Rows:", len(df))
display(df.head())

In [ ]:
# Shared field tools

RAW_TERMS = {
    "1": lambda r, c: np.ones_like(r),
    "c": lambda r, c: c,
    "r": lambda r, c: r,
    "c^3": lambda r, c: c**3,
    "r^2": lambda r, c: r**2,
    "r c^2": lambda r, c: r * c**2,
}
BASE_TERM_NAMES = list(RAW_TERMS.keys())

def raw_design_matrix(data, term_names):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return np.column_stack([RAW_TERMS[t](r, c) for t in term_names])

def centered_design_matrix(data, term_names, center_map=None):
    X = raw_design_matrix(data, term_names)
    if center_map is None:
        return X
    Xc = X.copy()
    for j, t in enumerate(term_names):
        Xc[:, j] = X[:, j] - center_map.get(t, 0.0)
    return Xc

def make_center_map(data, term_names):
    X = raw_design_matrix(data, term_names)
    return {t: float(X[:, j].mean()) for j, t in enumerate(term_names)}

def fit_template(sub, term_names, alpha=1e-6, center_map=None):
    X = centered_design_matrix(sub, term_names, center_map=center_map)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta
    stats = {"n": len(sub), "template_rmse": math.sqrt(mean_squared_error(y, pred))}
    for name, coef in zip(term_names, beta):
        stats[f"coef_{name}"] = float(coef)
    return beta, pred, stats

def predict_with_beta(sub, term_names, beta, center_map=None):
    return centered_design_matrix(sub, term_names, center_map=center_map) @ np.asarray(beta, dtype=float)

def trajectory_gap_beta(sub, term_names, beta_ref, beta_cmp, center_ref=None, center_cmp=None, n_r0=15):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate(beta, center_map, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i + 1] - cgrid[i])
            row = pd.DataFrame({"residual": [r], "condition_coord": [c]})
            g = float(np.clip(predict_with_beta(row, term_names, beta, center_map=center_map)[0], -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    return float(np.mean([math.sqrt(mean_squared_error(integrate(beta_ref, center_ref, r0), integrate(beta_cmp, center_cmp, r0))) for r0 in r0s]))

def build_coef_table(term_names, center_policy="raw"):
    group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]
    rows, subsets, center_maps = [], {}, {}

    for vals, sub in df.groupby(group_cols):
        if len(sub) < 30:
            continue

        if center_policy == "centered_constant":
            center_map = {"1": 1.0}
        elif center_policy == "centered_all":
            center_map = make_center_map(sub, term_names)
        else:
            center_map = None

        beta, pred, stats = fit_template(sub.copy(), term_names, center_map=center_map)
        kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
        regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
        row = {"regime_id": regime_id, "system": vals[0], "task": vals[1], "forcing_mode": vals[2], "k": float(vals[3]), "flow_mode": vals[4]}
        row.update(stats)
        rows.append(row)
        subsets[regime_id] = sub.copy()
        center_maps[regime_id] = center_map

    return pd.DataFrame(rows).reset_index(drop=True), subsets, center_maps

In [ ]:
# Metadata symbolic-chart helpers

def build_symbolic_features(df_in, columns=None, reduced_terms=None):
    X = pd.get_dummies(df_in[["forcing_mode", "flow_mode", "system", "task"]], drop_first=False)
    X["k"] = df_in["k"].astype(float).values
    X["k2"] = df_in["k"].astype(float).values ** 2

    ff = df_in["forcing_mode"].astype(str) + "__x__" + df_in["flow_mode"].astype(str)
    sf = df_in["system"].astype(str) + "__x__" + df_in["forcing_mode"].astype(str)
    X_ff = pd.get_dummies(ff, prefix="ff")
    X_sf = pd.get_dummies(sf, prefix="sf")
    X_fk = pd.get_dummies(df_in["forcing_mode"].astype(str), prefix="fk").multiply(df_in["k"].astype(float).to_numpy().reshape(-1, 1))

    X = pd.concat([X, X_ff, X_sf, X_fk], axis=1)
    if reduced_terms is not None:
        X = X.reindex(columns=reduced_terms, fill_value=0.0)
    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0.0)
    return X.astype(float)

def term_stability_table(coef_df, coef_cols, n_splits=8, threshold=1e-4):
    n = len(coef_df)
    splitter = LeaveOneOut() if n <= 12 else KFold(n_splits=min(n_splits, n), shuffle=True, random_state=42)
    all_features = build_symbolic_features(coef_df).columns.tolist()
    stability = {coef: {feat: 0 for feat in all_features} for coef in coef_cols}
    fold_count = 0

    for train_idx, _ in splitter.split(coef_df):
        train_df = coef_df.iloc[train_idx].reset_index(drop=True)
        X_train = build_symbolic_features(train_df, columns=all_features)
        for coef in coef_cols:
            y = train_df[coef].to_numpy(dtype=float)
            scaler = StandardScaler()
            Xs = scaler.fit_transform(X_train)
            model = LassoCV(cv=min(5, len(train_df)), random_state=fold_count + 1, max_iter=30000)
            model.fit(Xs, y)
            for feat, val in zip(all_features, model.coef_):
                if abs(val) > threshold:
                    stability[coef][feat] += 1
        fold_count += 1

    return pd.DataFrame([{"coefficient": coef, "term": feat, "frequency": stability[coef][feat] / max(fold_count, 1), "count": stability[coef][feat], "folds": fold_count} for coef in coef_cols for feat in all_features])

def stable_terms_by_coefficient(coef_df, coef_cols, threshold=0.5):
    stability_df = term_stability_table(coef_df, coef_cols)
    terms_by_coef = {}
    for coef in coef_cols:
        sub = stability_df[(stability_df["coefficient"] == coef) & (stability_df["frequency"] >= threshold)]
        terms_by_coef[coef] = sub["term"].tolist()
    return terms_by_coef, stability_df

def predict_pruned_coefficients(train_df, test_df, coef_cols, terms_by_coef):
    Yhat = np.zeros((len(test_df), len(coef_cols)), dtype=float)
    for j, coef in enumerate(coef_cols):
        terms = terms_by_coef.get(coef, [])
        if len(terms) == 0:
            Yhat[:, j] = train_df[coef].mean()
            continue
        X_train = build_symbolic_features(train_df, reduced_terms=terms)
        X_test = build_symbolic_features(test_df, columns=X_train.columns, reduced_terms=terms)
        y_train = train_df[coef].to_numpy(dtype=float)
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X_train)
        Xte = scaler.transform(X_test)
        model = Ridge(alpha=1.0)
        model.fit(Xtr, y_train)
        Yhat[:, j] = model.predict(Xte)
    return Yhat

hard_block_masks_template = {
    "k_high": lambda x: x["k"].astype(float) == 7.0,
    "k_low": lambda x: x["k"].astype(float) == 3.0,
    "forcing::condition": lambda x: x["forcing_mode"].astype(str) == "condition_gap",
    "forcing::capacity": lambda x: x["forcing_mode"].astype(str) == "capacity_gap",
    "forcing::feature": lambda x: x["forcing_mode"].astype(str) == "feature_gap",
    "system::entropy": lambda x: x["system"].astype(str) == "entropy",
    "system::unevenness": lambda x: x["system"].astype(str) == "unevenness",
    "flow::linear": lambda x: x["flow_mode"].astype(str) == "linear",
    "flow::nonlinear": lambda x: x["flow_mode"].astype(str) == "nonlinear",
}

def evaluate_basis(term_names, label, center_policy="raw"):
    coef_df_basis, subsets, center_maps = build_coef_table(term_names, center_policy=center_policy)
    coef_cols = [f"coef_{t}" for t in term_names]
    terms_by_coef, stability_df = stable_terms_by_coefficient(coef_df_basis, coef_cols, threshold=0.5)

    rows = []
    for block_name, mask_fn in hard_block_masks_template.items():
        test_mask = mask_fn(coef_df_basis)
        train_coef = coef_df_basis.loc[~test_mask].reset_index(drop=True)
        test_coef = coef_df_basis.loc[test_mask].reset_index(drop=True)
        if len(test_coef) == 0 or len(train_coef) < 5:
            continue

        beta_hat_mat = predict_pruned_coefficients(train_coef, test_coef, coef_cols, terms_by_coef)

        for i, rid in enumerate(test_coef["regime_id"].tolist()):
            sub = subsets[rid]
            beta_true = test_coef.loc[i, coef_cols].to_numpy(dtype=float)
            beta_hat = beta_hat_mat[i]
            y_true = sub["predicted_flow"].to_numpy(dtype=float)
            center = center_maps.get(rid, None)
            y_pred = predict_with_beta(sub, term_names, beta_hat, center_map=center)
            rows.append({
                "basis": label,
                "center_policy": center_policy,
                "block": block_name,
                "regime_id": rid,
                "n_terms": len(term_names),
                "coef_rmse": math.sqrt(mean_squared_error(beta_true, beta_hat)),
                "field_rmse": math.sqrt(mean_squared_error(y_true, y_pred)),
                "traj_rmse": trajectory_gap_beta(sub, term_names, beta_true, beta_hat, center_ref=center, center_cmp=center),
                "stable_chart_terms": sum(len(v) for v in terms_by_coef.values()),
            })

    return pd.DataFrame(rows), coef_df_basis, stability_df, terms_by_coef

## 4. Constant-term variants

In [ ]:
constant_variants = {
    "base_6_terms": (BASE_TERM_NAMES, "raw"),
    "no_constant": ([t for t in BASE_TERM_NAMES if t != "1"], "raw"),
    "centered_constant": (BASE_TERM_NAMES, "centered_constant"),
    "centered_all_terms": (BASE_TERM_NAMES, "centered_all"),
}

variant_frames = []
variant_details = {}

for label, (terms, policy) in constant_variants.items():
    print("Evaluating", label, terms, policy)
    eval_df, coef_table, stability_df, terms_by_coef = evaluate_basis(terms, label, center_policy=policy)
    variant_frames.append(eval_df)
    variant_details[label] = {"terms": terms, "policy": policy, "coef_table": coef_table, "stability_df": stability_df, "terms_by_coef": terms_by_coef}

variant_eval_df = pd.concat(variant_frames, ignore_index=True)
variant_summary_df = variant_eval_df.groupby("basis")[["coef_rmse", "field_rmse", "traj_rmse", "stable_chart_terms", "n_terms"]].mean().reset_index()
variant_summary_df = variant_summary_df.sort_values("field_rmse")
display(variant_summary_df)

variant_eval_df.to_csv(os.path.join(OUTPUT_DIR, "constant_variant_eval_raw.csv"), index=False)
variant_summary_df.to_csv(os.path.join(OUTPUT_DIR, "constant_variant_summary.csv"), index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_order = variant_summary_df.sort_values("field_rmse")["basis"].tolist()
for ax, metric in zip(axes, ["field_rmse", "traj_rmse"]):
    sub = variant_summary_df.set_index("basis").loc[plot_order].reset_index()
    ax.bar(sub["basis"], sub[metric])
    ax.set_title(f"Constant variant comparison — {metric}")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure1_constant_variant_comparison.png"), dpi=220, bbox_inches="tight")
plt.show()

## 5. Pairwise ablation

In [ ]:
PAIRWISE_REMOVALS = [
    ("c^3", "r^2"),
    ("c^3", "r c^2"),
    ("r^2", "r c^2"),
    ("1", "c^3"),
    ("1", "r^2"),
    ("1", "r c^2"),
    ("1", "r"),
    ("r", "r^2"),
]

pair_frames = []
for pair in PAIRWISE_REMOVALS:
    terms = [t for t in BASE_TERM_NAMES if t not in pair]
    label = "remove_pair::" + "+".join(pair)
    print("Evaluating", label, terms)
    eval_df, coef_table, stability_df, terms_by_coef = evaluate_basis(terms, label, center_policy="raw")
    eval_df["removed_pair"] = "+".join(pair)
    pair_frames.append(eval_df)

pair_eval_df = pd.concat(pair_frames, ignore_index=True)
pair_summary_df = pair_eval_df.groupby(["basis", "removed_pair"])[["coef_rmse", "field_rmse", "traj_rmse", "stable_chart_terms", "n_terms"]].mean().reset_index()

base_row = variant_summary_df[variant_summary_df["basis"] == "base_6_terms"].iloc[0]
pair_summary_df["field_delta"] = pair_summary_df["field_rmse"] - base_row["field_rmse"]
pair_summary_df["traj_delta"] = pair_summary_df["traj_rmse"] - base_row["traj_rmse"]
pair_summary_df["verdict"] = np.where(
    (pair_summary_df["field_delta"] <= 0) & (pair_summary_df["traj_delta"] <= 0),
    "candidate reduction",
    np.where((pair_summary_df["field_delta"] > 0) & (pair_summary_df["traj_delta"] > 0), "pair necessary", "mixed tradeoff"),
)

display(pair_summary_df.sort_values("field_delta"))
pair_eval_df.to_csv(os.path.join(OUTPUT_DIR, "pairwise_ablation_eval_raw.csv"), index=False)
pair_summary_df.to_csv(os.path.join(OUTPUT_DIR, "pairwise_ablation_summary.csv"), index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_df = pair_summary_df.sort_values("field_delta")
for ax, metric in zip(axes, ["field_delta", "traj_delta"]):
    ax.bar(plot_df["removed_pair"], plot_df[metric])
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(f"Pairwise ablation {metric}")
    ax.set_ylabel("ablated - base")
    ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure2_pairwise_ablation_deltas.png"), dpi=220, bbox_inches="tight")
plt.show()

## 6. Orthogonalized basis validation

In [ ]:
def fit_orthogonal_template(sub, term_names, alpha=1e-6):
    X = raw_design_matrix(sub, term_names)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    scaler = StandardScaler(with_mean=True, with_std=True)
    Xs = scaler.fit_transform(X)
    Q, R = np.linalg.qr(Xs)
    gamma = np.linalg.solve(Q.T @ Q + alpha * np.eye(Q.shape[1]), Q.T @ y)
    pred = Q @ gamma
    return gamma, pred, {"scaler": scaler, "R": R}

def build_orthogonal_coef_table(term_names):
    group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]
    rows, subsets, packs = [], {}, {}
    for vals, sub in df.groupby(group_cols):
        if len(sub) < 30:
            continue
        gamma, pred, pack = fit_orthogonal_template(sub.copy(), term_names)
        kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
        regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
        row = {"regime_id": regime_id, "system": vals[0], "task": vals[1], "forcing_mode": vals[2], "k": float(vals[3]), "flow_mode": vals[4], "template_rmse": math.sqrt(mean_squared_error(sub["predicted_flow"].to_numpy(dtype=float), pred))}
        for j in range(len(gamma)):
            row[f"coef_q{j+1}"] = float(gamma[j])
        rows.append(row)
        subsets[regime_id] = sub.copy()
        packs[regime_id] = pack
    return pd.DataFrame(rows), subsets, packs

def predict_orthogonal_field(sub, gamma, pack, term_names):
    X = raw_design_matrix(sub, term_names)
    Xs = pack["scaler"].transform(X)
    Q_new = Xs @ np.linalg.pinv(pack["R"])
    return Q_new @ gamma

def evaluate_orthogonal_basis(term_names):
    coef_df_o, subsets, packs = build_orthogonal_coef_table(term_names)
    coef_cols = [c for c in coef_df_o.columns if c.startswith("coef_q")]
    terms_by_coef, stability_df = stable_terms_by_coefficient(coef_df_o, coef_cols, threshold=0.5)

    rows = []
    for block_name, mask_fn in hard_block_masks_template.items():
        test_mask = mask_fn(coef_df_o)
        train_coef = coef_df_o.loc[~test_mask].reset_index(drop=True)
        test_coef = coef_df_o.loc[test_mask].reset_index(drop=True)
        if len(test_coef) == 0 or len(train_coef) < 5:
            continue
        gamma_hat_mat = predict_pruned_coefficients(train_coef, test_coef, coef_cols, terms_by_coef)
        for i, rid in enumerate(test_coef["regime_id"].tolist()):
            sub = subsets[rid]
            gamma_true = test_coef.loc[i, coef_cols].to_numpy(dtype=float)
            gamma_hat = gamma_hat_mat[i]
            y_true = sub["predicted_flow"].to_numpy(dtype=float)
            true_pred = predict_orthogonal_field(sub, gamma_true, packs[rid], term_names)
            pred = predict_orthogonal_field(sub, gamma_hat, packs[rid], term_names)
            rows.append({
                "basis": "orthogonalized_basis",
                "block": block_name,
                "regime_id": rid,
                "n_terms": len(term_names),
                "coef_rmse": math.sqrt(mean_squared_error(gamma_true, gamma_hat)),
                "field_rmse": math.sqrt(mean_squared_error(y_true, pred)),
                "traj_proxy_rmse": math.sqrt(mean_squared_error(true_pred, pred)),
                "stable_chart_terms": sum(len(v) for v in terms_by_coef.values()),
            })
    return pd.DataFrame(rows), coef_df_o, stability_df, terms_by_coef

ortho_eval_df, ortho_coef_df, ortho_stability_df, ortho_terms = evaluate_orthogonal_basis(BASE_TERM_NAMES)
ortho_summary_df = ortho_eval_df.groupby("basis")[["coef_rmse", "field_rmse", "traj_proxy_rmse", "stable_chart_terms", "n_terms"]].mean().reset_index()
display(ortho_summary_df)

ortho_eval_df.to_csv(os.path.join(OUTPUT_DIR, "orthogonalized_basis_eval_raw.csv"), index=False)
ortho_summary_df.to_csv(os.path.join(OUTPUT_DIR, "orthogonalized_basis_summary.csv"), index=False)

raw_for_ortho = variant_summary_df[variant_summary_df["basis"] == "base_6_terms"].copy()
ortho_compare_df = pd.DataFrame([
    {"basis": "raw_basis", "field_rmse": float(raw_for_ortho["field_rmse"].iloc[0]), "traj_metric": float(raw_for_ortho["traj_rmse"].iloc[0]), "stable_chart_terms": float(raw_for_ortho["stable_chart_terms"].iloc[0])},
    {"basis": "orthogonalized_basis", "field_rmse": float(ortho_summary_df["field_rmse"].iloc[0]), "traj_metric": float(ortho_summary_df["traj_proxy_rmse"].iloc[0]), "stable_chart_terms": float(ortho_summary_df["stable_chart_terms"].iloc[0])},
])
display(ortho_compare_df)
ortho_compare_df.to_csv(os.path.join(OUTPUT_DIR, "raw_vs_orthogonalized_comparison.csv"), index=False)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, metric in zip(axes, ["field_rmse", "traj_metric", "stable_chart_terms"]):
    ax.bar(ortho_compare_df["basis"], ortho_compare_df[metric])
    ax.set_title(metric)
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure3_raw_vs_orthogonalized.png"), dpi=220, bbox_inches="tight")
plt.show()

## 7. Final basis verdict

In [ ]:
base = variant_summary_df[variant_summary_df["basis"] == "base_6_terms"].iloc[0]
best_field_variant = variant_summary_df.sort_values("field_rmse").iloc[0]
best_traj_variant = variant_summary_df.sort_values("traj_rmse").iloc[0]

candidate_reductions = pair_summary_df[pair_summary_df["verdict"] == "candidate reduction"].copy()
orth_field_delta = float(ortho_compare_df[ortho_compare_df["basis"] == "orthogonalized_basis"]["field_rmse"].iloc[0] - base["field_rmse"])
orth_stability_delta = float(ortho_compare_df[ortho_compare_df["basis"] == "orthogonalized_basis"]["stable_chart_terms"].iloc[0] - base["stable_chart_terms"])

if best_field_variant["basis"] != "base_6_terms" and best_field_variant["field_rmse"] <= base["field_rmse"] * 1.01:
    constant_verdict = f"prefer {best_field_variant['basis']} for field RMSE"
elif best_traj_variant["basis"] != "base_6_terms" and best_traj_variant["traj_rmse"] <= base["traj_rmse"] * 0.98:
    constant_verdict = f"{best_traj_variant['basis']} improves trajectories but should be reported as trajectory-safe variant"
else:
    constant_verdict = "retain raw base_6_terms; constant-term variants do not improve enough to replace raw field"

if len(candidate_reductions) > 0:
    pairwise_verdict = "pairwise reduction candidate exists; inspect before final paper"
else:
    pairwise_verdict = "no pairwise reduction improves both field and trajectory; six-term basis remains preferred"

if orth_field_delta <= 0 and orth_stability_delta <= 0:
    ortho_verdict = "orthogonalization improves or preserves RMSE and reduces chart complexity; include as appendix validation"
elif orth_stability_delta < 0:
    ortho_verdict = "orthogonalization reduces chart complexity but does not replace raw interpretable equation"
else:
    ortho_verdict = "orthogonalization does not improve final model; keep raw basis"

if ("retain raw" in constant_verdict) and ("six-term basis remains preferred" in pairwise_verdict):
    final_verdict = "raw six-term field remains final model"
elif len(candidate_reductions) > 0:
    final_verdict = "reduced basis candidate needs follow-up before final paper"
else:
    final_verdict = "six-term field remains final, with variant note in appendix"

verdict_df = pd.DataFrame([{
    "base_field_rmse": float(base["field_rmse"]),
    "base_traj_rmse": float(base["traj_rmse"]),
    "best_field_variant": best_field_variant["basis"],
    "best_field_rmse": float(best_field_variant["field_rmse"]),
    "best_traj_variant": best_traj_variant["basis"],
    "best_traj_rmse": float(best_traj_variant["traj_rmse"]),
    "n_pairwise_reduction_candidates": int(len(candidate_reductions)),
    "orthogonal_field_delta": orth_field_delta,
    "orthogonal_stability_delta": orth_stability_delta,
    "constant_verdict": constant_verdict,
    "pairwise_verdict": pairwise_verdict,
    "orthogonal_verdict": ortho_verdict,
    "final_verdict": final_verdict,
}])

display(verdict_df)
verdict_df.to_csv(os.path.join(OUTPUT_DIR, "final_basis_verdict.csv"), index=False)

## 8. Paper-ready paragraph and manifest

In [ ]:
paragraph = f'''
## Minimal basis refinement

We tested whether the final six-term field required centering, pairwise reduction, or orthogonalized coordinates. Constant-term variants showed: `{constant_verdict}`. Pairwise ablation showed: `{pairwise_verdict}`. Orthogonalized-basis validation showed: `{ortho_verdict}`. The final basis verdict is: `{final_verdict}`. These tests separate interpretability from numerical conditioning: the raw six-term equation remains the paper-facing governing field unless a reduced pair passes both field and trajectory criteria, while orthogonalized coordinates can be reported as validation for basis correlation rather than as a replacement equation.
'''
with open(os.path.join(OUTPUT_DIR, "minimal_basis_refinement_paragraph.md"), "w", encoding="utf-8") as f:
    f.write(paragraph.strip() + "\\n")
display(Markdown(paragraph))

manifest = {
    "data_source": data_source,
    "synthetic_fallback": bool(USE_SYNTHETIC_FALLBACK),
    "base_terms": BASE_TERM_NAMES,
    "constant_variants": list(constant_variants.keys()),
    "pairwise_removals": ["+".join(p) for p in PAIRWISE_REMOVALS],
    "final_verdict": final_verdict,
    "outputs": sorted(os.listdir(OUTPUT_DIR)),
}
with open(os.path.join(OUTPUT_DIR, "manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

display(pd.DataFrame({"output_file": sorted(os.listdir(OUTPUT_DIR))}))
print("Saved outputs under:", OUTPUT_DIR)

## Final statement

Notebook 69 is not a new modeling branch. It answers the last basis-level question:

> Is the raw six-term symbolic field final, or should it be centered/reduced/orthogonalized?

Use its final verdict to decide whether the paper-facing equation remains the raw six-term field.